In [1]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/TiendaBigData.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 120 registros.
+--------------------+-------------------+-------------------+--------------------+-----+
|                 _id|      fecha_captura|              grupo|       identificador|valor|
+--------------------+-------------------+-------------------+--------------------+-----+
|69e7aff00c92ca8d7...|2026-04-21 17:11:29|Turismo y hoteleria|15.6" FHD Laptop ...|399.0|
|69e7aff00c92ca8d7...|2026-04-21 17:11:30|Turismo y hoteleria|Laptop 15.6 Inch ...|349.0|
|69e7aff00c92ca8d7...|2026-04-21 17:11:30|Turismo y hoteleria|Laptop 15.6 Pouce...|349.0|
|69e7aff00c92ca8d7...|2026-04-21 17:11:30|Turismo y hoteleria|Laptop 14 inch Du...|205.0|
|69e7aff00c92ca8d7...|2026-04-21 17:11:30|Turismo y hoteleria|15.6 Inch Laptop,...|289.0|
+--------------------+-------------------+-------------------+--------------------+-----+
only showing top 5 rows



In [2]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+------------------+
|    marca|   precio_promedio|
+---------+------------------+
|       LG|            1474.0|
|Microsoft|            1219.0|
|      MSI|1044.3333333333333|
|   Lenovo| 802.3636363636364|
|     ASUS| 781.7857142857143|
|     Acer| 780.3333333333334|
|     DELL|             749.0|
|     acer|             644.0|
|  Lenovo,|             579.0|
| ACEMAGIC| 561.8571428571429|
|       16|             439.0|
|    Ryzen|             424.0|
|    15.6"|             399.0|
|     Dell|             385.0|
|   Laptop|362.35714285714283|
|DREAMFYRE|             339.0|
|       HP| 338.8333333333333|
|     15.6|326.77777777777777|
|  Molegar|             319.0|
|   KEFEYA|             309.0|
+---------+------------------+
only showing top 20 rows

